## Training notebook

In [1]:
import gym
import highway_env
import datetime

from stable_baselines3 import PPO
# from sb3_contrib import RecurrentPPO

%load_ext tensorboard
import datetime, os
from ipywidgets import interact, widgets
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from typing import Callable

### Select operating system

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UNIX', 'WINDOWS'])

interactive(children=(Dropdown(description='desired_os', options=('UNIX', 'WINDOWS'), value='UNIX'), Output())…

<function __main__.f(desired_os)>

In [3]:
OUTPUT_PATH = '/Users/fornerispighetti/HighwayDRL/training_output/' if os_id.value == "UNIX" else 'C:/Users/aless/OneDrive/Desktop/Thesis_repo_new/HighwayDRL/training_output/'

In [4]:
# Create and wrap the environment
env = gym.make("multipleO-dm-env-v0")

# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=False, render=True)

In [4]:
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func

In [6]:
# PPO parameters
model = PPO('MlpPolicy', env, tensorboard_log=f'{OUTPUT_PATH}logdir', learning_rate=linear_schedule(3e-4), verbose=1)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [7]:
num_timesteps = 2e4

# Train the agent for n steps
model.learn(total_timesteps=int(num_timesteps), callback=[checkpoint_callback, eval_callback])

#Save the trained agent
model.save(f'{OUTPUT_PATH}models/ppo_RL_{str(os_id.value)}_2e4_{datetime.datetime.today().day}-{datetime.datetime.today().month}-{datetime.datetime.today().year}')

Logging to /Users/fornerispighetti/HighwayDRL/training_output/logdir/PPO_13


/Users/fornerispighetti/HighwayDRL/.venv/lib/python3.8/site-packages/stable_baselines3/common/evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=30.10 +/- 42.19
Episode length: 38.20 +/- 42.46
---------------------------------
| eval/              |          |
|    mean_ep_length  | 38.2     |
|    mean_reward     | 30.1     |
| time/              |          |
|    total_timesteps | 1000     |
---------------------------------
New best mean reward!


/Users/fornerispighetti/HighwayDRL/.venv/lib/python3.8/site-packages/stable_baselines3/common/evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=2000, episode_reward=16.32 +/- 9.35
Episode length: 31.20 +/- 21.78
---------------------------------
| eval/              |          |
|    mean_ep_length  | 31.2     |
|    mean_reward     | 16.3     |
| time/              |          |
|    total_timesteps | 2000     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 43.8     |
|    ep_rew_mean     | 33.9     |
| time/              |          |
|    fps             | 33       |
|    iterations      | 1        |
|    time_elapsed    | 61       |
|    total_timesteps | 2048     |
---------------------------------
Eval num_timesteps=3000, episode_reward=38.51 +/- 31.22
Episode length: 56.80 +/- 33.47
-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 56.8        |
|    mean_reward          | 38.5        |
| time/                   |             |
|    total_timesteps      | 3000     

### Continued learning

In [5]:
env_cl = gym.make("jam-dm-env-v0")

In [6]:
model_cl = PPO.load("/Users/fornerispighetti/HighwayDRL/training_output/models/ppo_RL_UNIX_4e4_8_6_2022_cl", env=env_cl, tensorboard_log=f"{OUTPUT_PATH}logdir")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [7]:
# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=False, render=True)

In [8]:
model_cl.learn(total_timesteps=int(2e5), reset_num_timesteps=False, callback=[checkpoint_callback, eval_callback])

Logging to /Users/fornerispighetti/HighwayDRL/training_output/logdir/PPO_13
